# 0050 AI Signal Quality Validation - FIXED

Recommended input: `0050_v1_1_ai_event_dataset_full_features.csv`.

This version fixes data_split normalization such as `Train_探索_2022_2024`.


In [1]:
# -*- coding: utf-8 -*-
"""
0050 AI Signal Quality Validation - FIXED

This notebook/script is an AI-assisted validation layer for the 0050 vs TW50
intraday deviation reversion strategy.

This is NOT an AI trading model.
It uses ML to validate whether entry-time features explain signal quality.

Recommended input:
    0050_v1_1_ai_event_dataset_full_features.csv

Output:
    0050_ai_signal_quality_validation_outputs_FIXED.xlsx

Main fixes in this version:
1. Normalize data_split names such as Train_探索_2022_2024 into Train_2022_2024.
2. Avoid pandas groupby.apply reset_index duplicate-column errors.
3. Add clear diagnostics if train / validation / recent split becomes empty.
4. Use time-based split instead of random split.
5. Exclude post-trade leakage fields such as exit_reason, MFE, MAE, holding_minutes.
"""

import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd

from IPython.display import display
from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance


# ============================================================
# 0. Parameters
# ============================================================

OUTPUT_FILE = "0050_ai_signal_quality_validation_outputs_FIXED.xlsx"
ROUND_TRIP_COST = 0.02
STRONG_EDGE_THRESHOLD = 0.05
RANDOM_STATE = 42


# ============================================================
# 1. Helper functions
# ============================================================

def clean_number(x):
    """Convert messy numeric text into float."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    s = s.replace(",", "")
    s = s.replace("TWD", "")
    s = s.replace("%", "")
    s = s.replace("−", "-")
    s = re.sub(r"[^\d\.\-\+eE]", "", s)
    if s in ["", "-", "+", "."]:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def find_col(df, candidates):
    """Find column by exact, lowercase, or fuzzy matching."""
    cols = list(df.columns)
    lower_map = {str(c).lower(): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand

    for cand in candidates:
        cand_low = cand.lower()
        if cand_low in lower_map:
            return lower_map[cand_low]

    for cand in candidates:
        cand_low = cand.lower()
        for c in cols:
            if cand_low in str(c).lower():
                return c

    return None


def normalize_time_hhmm(x):
    """Normalize time into HH:MM."""
    if pd.isna(x):
        return np.nan
    if hasattr(x, "strftime"):
        return x.strftime("%H:%M")

    s = str(x).strip()

    if re.fullmatch(r"\d{3,4}", s):
        s = s.zfill(4)
        return s[:2] + ":" + s[2:]

    m = re.search(r"(\d{1,2}):(\d{2})", s)
    if m:
        return f"{int(m.group(1)):02d}:{int(m.group(2)):02d}"

    return np.nan


def time_to_minutes(hhmm):
    if pd.isna(hhmm):
        return np.nan
    try:
        hh, mm = str(hhmm).split(":")[:2]
        return int(hh) * 60 + int(mm)
    except Exception:
        return np.nan


def assign_session(mins):
    if pd.isna(mins):
        return "Unknown"
    if mins < 9 * 60 + 5:
        return "pre_0905"
    elif mins < 9 * 60 + 45:
        return "early_main_0905_0945"
    elif mins < 10 * 60:
        return "early_mid_0945_1000"
    elif mins < 12 * 60 + 45:
        return "midday_main_1000_1245"
    elif mins <= 13 * 60:
        return "lunch_tail_1245_1300"
    else:
        return "after_1300"


def normalize_data_split(x, entry_year=None):
    """Normalize split labels, including Chinese labels in the current dataset."""
    s = "" if pd.isna(x) else str(x)

    if ("2022" in s and "2024" in s) or ("Train" in s) or ("探索" in s):
        return "Train_2022_2024"
    if ("2025" in s) or ("Validation" in s) or ("驗證" in s):
        return "Validation_2025"
    if ("2026" in s) or ("Recent" in s) or ("Check" in s) or ("近期" in s):
        return "Recent_Check_2026"

    if entry_year is not None and not pd.isna(entry_year):
        y = int(entry_year)
        if y <= 2024:
            return "Train_2022_2024"
        if y == 2025:
            return "Validation_2025"
        if y >= 2026:
            return "Recent_Check_2026"

    return "Unknown"


def calc_profit_factor(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gp = pnl[pnl > 0].sum()
    gl = pnl[pnl < 0].sum()
    if gl < 0:
        return gp / abs(gl)
    if gp > 0 and gl == 0:
        return np.inf
    return np.nan


def perf_summary(data, pnl_col="pnl_twd", group_cols=None):
    """Robust performance summary. Avoids groupby.apply/reset_index duplicate-column bugs."""

    def _calc(x):
        pnl = pd.to_numeric(x[pnl_col], errors="coerce").dropna()
        trades = len(pnl)
        if trades == 0:
            return {
                "trades": 0,
                "wins": 0,
                "losses": 0,
                "flats": 0,
                "win_rate": np.nan,
                "net_pnl": 0.0,
                "avg_pnl": np.nan,
                "gross_profit": 0.0,
                "gross_loss": 0.0,
                "profit_factor": np.nan,
            }
        gp = pnl[pnl > 0].sum()
        gl = pnl[pnl < 0].sum()
        return {
            "trades": int(trades),
            "wins": int((pnl > 0).sum()),
            "losses": int((pnl < 0).sum()),
            "flats": int((pnl == 0).sum()),
            "win_rate": float((pnl > 0).mean()),
            "net_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "gross_profit": float(gp),
            "gross_loss": float(gl),
            "profit_factor": calc_profit_factor(pnl),
        }

    if group_cols is None:
        return pd.DataFrame([_calc(data)])

    if isinstance(group_cols, str):
        group_cols = [group_cols]

    rows = []
    if len(data) == 0:
        return pd.DataFrame(columns=group_cols + list(_calc(data).keys()))

    for keys, g in data.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: val for col, val in zip(group_cols, keys)}
        row.update(_calc(g))
        rows.append(row)

    return pd.DataFrame(rows)


def safe_auc(y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_prob)
    except Exception:
        return np.nan


def evaluate_model(name, model, X, y, dataset_name):
    if len(y) == 0:
        return {
            "model": name,
            "dataset": dataset_name,
            "n": 0,
            "positive_rate": np.nan,
            "auc": np.nan,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan,
        }

    pred = model.predict(X)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X)[:, 1]
    else:
        prob = pred

    cm = confusion_matrix(y, pred, labels=[0, 1])

    return {
        "model": name,
        "dataset": dataset_name,
        "n": int(len(y)),
        "positive_rate": float(np.mean(y)),
        "auc": safe_auc(y, prob),
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }


def get_preprocessed_feature_names(preprocessor):
    names = []
    try:
        num_features = preprocessor.transformers_[0][2]
        cat_features = preprocessor.transformers_[1][2]
        names.extend(list(num_features))
        ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        cat_names = list(ohe.get_feature_names_out(cat_features))
        names.extend(cat_names)
    except Exception:
        pass
    return names


def display_round(df, title=None, n=None):
    out = df.copy()
    float_cols = out.select_dtypes(include=[float]).columns
    out[float_cols] = out[float_cols].round(6)
    if title:
        print("\n" + "=" * 90)
        print(title)
        print("=" * 90)
    if n:
        display(out.head(n))
    else:
        display(out)
    return out


# ============================================================
# 2. Load file
# ============================================================

print("Please upload the event dataset.")
print("Recommended: 0050_v1_1_ai_event_dataset_full_features.csv")
print("Acceptable: xlsx with Combined_Data sheet.")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print("Loaded:", file_name)

if file_name.lower().endswith(".csv"):
    raw = pd.read_csv(file_name)
elif file_name.lower().endswith(".xlsx"):
    xls = pd.ExcelFile(file_name)
    print("Sheets:", xls.sheet_names)
    if "Combined_Data" in xls.sheet_names:
        raw = pd.read_excel(file_name, sheet_name="Combined_Data")
    elif "Parsed_B3_Events" in xls.sheet_names:
        raw = pd.read_excel(file_name, sheet_name="Parsed_B3_Events")
    else:
        raw = pd.read_excel(file_name, sheet_name=xls.sheet_names[0])
else:
    raise ValueError("Please upload CSV or XLSX.")

print("Raw shape:", raw.shape)
print("Columns:")
print(raw.columns.tolist())
display(raw.head(3))

df = raw.copy()

# If Combined_Data contains multiple versions, prefer v1-like rows for original candidate pool.
version_col = find_col(df, ["version"])
if version_col is not None:
    unique_versions = df[version_col].dropna().astype(str).unique().tolist()
    print("Versions found:", unique_versions)
    v1_mask = df[version_col].astype(str).str.contains("v1", case=False, na=False)
    if v1_mask.sum() > 0:
        df = df[v1_mask].copy()
        print("Filtered to v1-like rows for baseline candidate pool:", df.shape)


# ============================================================
# 3. Standardize columns
# ============================================================

pnl_col = find_col(df, ["pnl_twd", "pnl_no_cost", "net_pnl", "淨損益 TWD", "pnl"])
if pnl_col is None:
    raise ValueError("Cannot find pnl column. Need pnl_twd or pnl_no_cost.")

df["pnl_twd"] = df[pnl_col].apply(clean_number)
df["pnl_cost_002"] = df["pnl_twd"] - ROUND_TRIP_COST

entry_dt_col = find_col(df, ["entry_dt", "entry_datetime", "entry_time", "entry_timestamp", "entry_date_time"])
date_col = find_col(df, ["entry_date", "date"])
hhmm_col = find_col(df, ["entry_time_hhmm", "entry_hhmm", "time_hhmm"])

if entry_dt_col is not None:
    df["entry_dt"] = pd.to_datetime(df[entry_dt_col], errors="coerce")
elif date_col is not None and hhmm_col is not None:
    df["entry_dt"] = pd.to_datetime(df[date_col].astype(str) + " " + df[hhmm_col].astype(str), errors="coerce")
else:
    df["entry_dt"] = pd.NaT

if "entry_year" not in df.columns:
    df["entry_year"] = df["entry_dt"].dt.year
df["entry_year"] = pd.to_numeric(df["entry_year"], errors="coerce")

if "entry_month" not in df.columns:
    df["entry_month"] = df["entry_dt"].dt.to_period("M").astype(str)

if "entry_weekday" not in df.columns:
    df["entry_weekday"] = df["entry_dt"].dt.day_name()

if hhmm_col is not None:
    df["entry_time_hhmm"] = df[hhmm_col].apply(normalize_time_hhmm)
else:
    df["entry_time_hhmm"] = df["entry_dt"].apply(normalize_time_hhmm)

df["entry_minutes"] = df["entry_time_hhmm"].apply(time_to_minutes)

session_col = find_col(df, ["session_block", "session"])
if session_col is not None:
    df["session_block"] = df[session_col].astype(str)
else:
    df["session_block"] = df["entry_minutes"].apply(assign_session)

sig_col = find_col(df, ["signal_count_today", "trade_count_today", "daily_signal_count"])
if sig_col is not None:
    df["signal_count_today"] = pd.to_numeric(df[sig_col], errors="coerce")
else:
    df = df.sort_values(["entry_dt"])
    df["signal_count_today"] = df.groupby(df["entry_dt"].dt.date).cumcount() + 1

# Normalize split robustly.
if "data_split" not in df.columns:
    df["data_split"] = df["entry_year"].apply(lambda y: normalize_data_split(None, y))
else:
    df["data_split_original"] = df["data_split"].astype(str)
    df["data_split"] = df.apply(lambda r: normalize_data_split(r.get("data_split"), r.get("entry_year")), axis=1)

print("Normalized data_split counts:")
print(df["data_split"].value_counts(dropna=False))

# Labels
df["good_signal_cost002"] = (df["pnl_cost_002"] > 0).astype(int)

def quality_label(p):
    if pd.isna(p):
        return "unknown"
    if p >= STRONG_EDGE_THRESHOLD:
        return "strong"
    if p > 0:
        return "weak"
    return "failed"

df["signal_quality_3class"] = df["pnl_cost_002"].apply(quality_label)


# ============================================================
# 4. Feature definition and leakage control
# ============================================================

candidate_numeric_features = [
    "entry_relative_deviation",
    "deviation_slope_1bar",
    "deviation_slope_3bar",
    "tw50_return_5m",
    "tw50_return_15m",
    "etf_return_5m",
    "etf_return_15m",
    "volume_ratio_at_entry",
    "realized_vol_10bar",
    "signal_count_today",
    "entry_minutes",
    "minutes_since_open",
    "minutes_to_close",
]

candidate_categorical_features = [
    "session_block",
    "entry_weekday",
]

# Exclude post-trade or label-derived fields.
leakage_keywords = [
    "exit", "mfe", "mae", "holding", "bars_held", "pnl",
    "profit", "quality_label", "signal_quality", "good_signal",
    "result", "label"
]

numeric_features = []
for c in candidate_numeric_features:
    if c in df.columns and not any(k.lower() in c.lower() for k in leakage_keywords):
        df[c] = pd.to_numeric(df[c], errors="coerce")
        numeric_features.append(c)

categorical_features = []
for c in candidate_categorical_features:
    if c in df.columns and not any(k.lower() in c.lower() for k in leakage_keywords):
        df[c] = df[c].astype(str)
        categorical_features.append(c)

feature_cols = numeric_features + categorical_features

if len(feature_cols) == 0:
    raise ValueError("No usable entry-time features found. Please check event dataset columns.")

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Model dataset
model_df = df.dropna(subset=["pnl_twd", "pnl_cost_002", "good_signal_cost002"]).copy()
valid_splits = ["Train_2022_2024", "Validation_2025", "Recent_Check_2026"]
model_df = model_df[model_df["data_split"].isin(valid_splits)].copy()

print("After split filter:", model_df.shape)
print(model_df["data_split"].value_counts(dropna=False))

if len(model_df) == 0:
    raise ValueError(
        "model_df is empty after data_split filtering. "
        "Please check data_split values and normalization."
    )

train_df = model_df[model_df["data_split"] == "Train_2022_2024"].copy()
valid_df = model_df[model_df["data_split"] == "Validation_2025"].copy()
recent_df = model_df[model_df["data_split"] == "Recent_Check_2026"].copy()

split_sizes = {
    "train": len(train_df),
    "validation": len(valid_df),
    "recent": len(recent_df),
}
print("Split sizes:", split_sizes)

if len(train_df) == 0:
    raise ValueError("Train set is empty. Cannot train model.")

X_train = train_df[feature_cols]
y_train = train_df["good_signal_cost002"]

X_valid = valid_df[feature_cols]
y_valid = valid_df["good_signal_cost002"]

X_recent = recent_df[feature_cols]
y_recent = recent_df["good_signal_cost002"]

if len(np.unique(y_train)) < 2:
    raise ValueError(
        "Train set contains only one class for good_signal_cost002. "
        "Model training would be invalid."
    )


# ============================================================
# 5. Data diagnostics and distributions
# ============================================================

data_check = pd.DataFrame({
    "item": [
        "input_file",
        "raw_rows",
        "rows_after_filter",
        "train_rows",
        "validation_rows",
        "recent_rows",
        "pnl_col",
        "round_trip_cost_for_label",
        "positive_label_definition",
        "numeric_features",
        "categorical_features",
        "leakage_excluded",
        "data_split_values_after_normalization"
    ],
    "value": [
        file_name,
        len(raw),
        len(model_df),
        len(train_df),
        len(valid_df),
        len(recent_df),
        pnl_col,
        ROUND_TRIP_COST,
        "good_signal_cost002 = 1 if pnl_twd - 0.02 > 0",
        ", ".join(numeric_features),
        ", ".join(categorical_features),
        "exit_reason, exit price, exit deviation, MFE, MAE, holding time, pnl-derived labels",
        str(model_df["data_split"].value_counts(dropna=False).to_dict())
    ]
})

label_distribution = (
    model_df.groupby(["data_split", "signal_quality_3class"])
    .size()
    .reset_index(name="count")
)

binary_label_distribution = (
    model_df.groupby(["data_split", "good_signal_cost002"])
    .size()
    .reset_index(name="count")
)

overall_perf = perf_summary(model_df, "pnl_twd")
perf_by_split = perf_summary(model_df, "pnl_twd", "data_split")
perf_by_quality = perf_summary(model_df, "pnl_twd", "signal_quality_3class")
quality_by_session = perf_summary(model_df, "pnl_twd", ["session_block", "signal_quality_3class"])

# Feature missingness
missingness = pd.DataFrame({
    "feature": feature_cols,
    "missing_rate": [model_df[c].isna().mean() for c in feature_cols],
    "dtype": [str(model_df[c].dtype) for c in feature_cols]
}).sort_values("missing_rate", ascending=False)

display_round(data_check, "Data Check")
display_round(label_distribution, "Three-Class Label Distribution")
display_round(binary_label_distribution, "Binary Label Distribution")
display_round(perf_by_split, "Performance by Split")
display_round(missingness, "Feature Missingness")


# ============================================================
# 6. Model pipelines
# ============================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

models = {}

models["Logistic_Regression"] = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
])

models["Random_Forest"] = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=4,
        min_samples_leaf=8,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample"
    ))
])

# Optional XGBoost
xgb_available = False
try:
    from xgboost import XGBClassifier
    models["XGBoost_Optional"] = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE
        ))
    ])
    xgb_available = True
except Exception as e:
    print("XGBoost not available. Skipping XGBoost.", e)


# ============================================================
# 7. Train and evaluate
# ============================================================

metrics_rows = []
feature_importance_tables = {}
trained_models = {}

for name, model in models.items():
    print("\nTraining:", name)
    model.fit(X_train, y_train)
    trained_models[name] = model

    metrics_rows.append(evaluate_model(name, model, X_train, y_train, "Train_2022_2024"))

    if len(valid_df) > 0:
        metrics_rows.append(evaluate_model(name, model, X_valid, y_valid, "Validation_2025"))

    if len(recent_df) > 0:
        metrics_rows.append(evaluate_model(name, model, X_recent, y_recent, "Recent_Check_2026"))

    try:
        preprocess_fitted = model.named_steps["preprocess"]
        feat_names = get_preprocessed_feature_names(preprocess_fitted)
        final_model = model.named_steps["model"]

        if hasattr(final_model, "feature_importances_"):
            values = final_model.feature_importances_
            fi = pd.DataFrame({
                "feature": feat_names[:len(values)],
                "importance": values
            }).sort_values("importance", ascending=False)
            feature_importance_tables[name] = fi

        elif hasattr(final_model, "coef_"):
            values = final_model.coef_[0]
            fi = pd.DataFrame({
                "feature": feat_names[:len(values)],
                "coef": values,
                "abs_coef": np.abs(values)
            }).sort_values("abs_coef", ascending=False)
            feature_importance_tables[name] = fi

    except Exception as e:
        print("Feature importance failed for", name, ":", e)

metrics_df = pd.DataFrame(metrics_rows)

display_round(metrics_df, "Model Metrics")

for name, fi in feature_importance_tables.items():
    display_round(fi, f"Feature Importance - {name}", n=20)


# ============================================================
# 8. Permutation importance on validation
# ============================================================

permutation_tables = {}

if len(valid_df) > 10 and len(np.unique(y_valid)) >= 2:
    for name, model in trained_models.items():
        try:
            print("Permutation importance on validation:", name)
            result = permutation_importance(
                model,
                X_valid,
                y_valid,
                n_repeats=20,
                random_state=RANDOM_STATE,
                scoring="roc_auc"
            )
            perm = pd.DataFrame({
                "feature": feature_cols,
                "importance_mean": result.importances_mean,
                "importance_std": result.importances_std
            }).sort_values("importance_mean", ascending=False)
            permutation_tables[name] = perm
            display_round(perm, f"Permutation Importance - {name}", n=20)
        except Exception as e:
            print("Permutation importance failed for", name, ":", e)
else:
    print("Validation set too small or one-class only. Skipping permutation importance.")


# ============================================================
# 9. SHAP optional
# ============================================================

shap_summary = pd.DataFrame({
    "status": ["not_run"],
    "reason": ["SHAP is optional. Use feature importance and permutation importance as baseline."]
})

try:
    import shap
    if "Random_Forest" in trained_models and len(valid_df) > 0:
        rf_pipe = trained_models["Random_Forest"]
        preprocess_fitted = rf_pipe.named_steps["preprocess"]
        rf_model = rf_pipe.named_steps["model"]
        X_valid_trans = preprocess_fitted.transform(X_valid)
        feat_names = get_preprocessed_feature_names(preprocess_fitted)
        try:
            X_valid_dense = X_valid_trans.toarray()
        except Exception:
            X_valid_dense = X_valid_trans
        explainer = shap.TreeExplainer(rf_model)
        shap_values = explainer.shap_values(X_valid_dense)
        if isinstance(shap_values, list):
            shap_vals = shap_values[1]
        else:
            shap_vals = shap_values
        mean_abs_shap = np.abs(shap_vals).mean(axis=0)
        shap_summary = pd.DataFrame({
            "feature": feat_names[:len(mean_abs_shap)],
            "mean_abs_shap": mean_abs_shap
        }).sort_values("mean_abs_shap", ascending=False)
        display_round(shap_summary, "SHAP Summary - Random Forest", n=20)
    else:
        shap_summary = pd.DataFrame({
            "status": ["not_run"],
            "reason": ["Random Forest model or validation set not available."]
        })
except Exception as e:
    shap_summary = pd.DataFrame({
        "status": ["not_run"],
        "reason": [str(e)]
    })
    print("SHAP skipped:", e)


# ============================================================
# 10. Score bucket analysis
# ============================================================

score_tables = {}
scored_datasets = []

for name, model in trained_models.items():
    temp_frames = []
    for dataset_name, part_df, X_part in [
        ("Train_2022_2024", train_df, X_train),
        ("Validation_2025", valid_df, X_valid),
        ("Recent_Check_2026", recent_df, X_recent),
    ]:
        if len(part_df) == 0:
            continue

        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(X_part)[:, 1]
        else:
            prob = model.predict(X_part)

        temp = part_df.copy()
        temp["model"] = name
        temp["dataset"] = dataset_name
        temp["pred_good_prob"] = prob

        try:
            temp["score_bucket"] = pd.qcut(temp["pred_good_prob"], q=5, duplicates="drop").astype(str)
        except Exception:
            temp["score_bucket"] = pd.cut(temp["pred_good_prob"], bins=5).astype(str)

        temp_frames.append(temp)

    if temp_frames:
        scored = pd.concat(temp_frames, ignore_index=True)
        scored_datasets.append(scored)
        bucket_perf = perf_summary(scored, "pnl_cost_002", ["model", "dataset", "score_bucket"])
        score_tables[name] = bucket_perf
        display_round(bucket_perf, f"Score Bucket Performance - {name}", n=30)

if scored_datasets:
    scored_all = pd.concat(scored_datasets, ignore_index=True)
else:
    scored_all = pd.DataFrame()


# ============================================================
# 11. Conclusion
# ============================================================

top_features_text = []
for name, fi in feature_importance_tables.items():
    top_col = "feature"
    top_feats = fi.head(8)[top_col].astype(str).tolist()
    top_features_text.append(f"{name}: " + ", ".join(top_feats))

if not top_features_text:
    top_features_text.append("No feature importance available.")

conclusion_text = f"""
0050 AI Signal Quality Validation Conclusion

1. This module is an AI-assisted validation layer, not an AI trading model.
2. The target label is good_signal_cost002 = 1 if pnl_twd - {ROUND_TRIP_COST:.2f} > 0.
3. The purpose is to test whether entry-time features can explain signal quality after cost.
4. We used time-based split instead of random split:
   Train = 2022-2024, Validation = 2025, Recent Check = 2026.
5. Leakage columns are intentionally excluded, including exit_reason, exit_price,
   exit_relative_deviation, MFE, MAE, holding time, and pnl-derived labels.
6. Main models:
   Logistic Regression for interpretable baseline,
   Random Forest for non-linear interaction check,
   XGBoost optional if the environment supports it.
7. Important interpretation:
   If TW50_return_5m, deviation slope, session, signal_count_today,
   realized_vol_10bar, or volume_ratio_at_entry appear important,
   this supports the original market-logic research.
8. This module should be presented as:
   AI-driven signal quality validation / feature stability check,
   not as an autonomous trading model.

Top feature snapshots:
{chr(10).join(top_features_text)}
"""

conclusion_df = pd.DataFrame({"conclusion": [conclusion_text]})

readme_df = pd.DataFrame({
    "section": [
        "Purpose",
        "Input",
        "Label",
        "Leakage Control",
        "Modeling Principle",
        "How to use in portfolio"
    ],
    "content": [
        "Use machine learning to validate signal quality features in the 0050 vs TW50 event dataset.",
        "Recommended input: v1.1 event dataset, because it is closer to the original candidate signal pool.",
        "good_signal_cost002 = 1 if pnl_twd - 0.02 > 0. Three-class label is strong / weak / failed.",
        "Exclude post-trade columns such as exit_reason, MFE, MAE, exit price, holding time, and pnl-derived features.",
        "Use time-based split. Do not use random split for financial event validation.",
        "Present this as AI-assisted validation, not as an AI trading model."
    ]
})


# ============================================================
# 12. Export Excel
# ============================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    readme_df.to_excel(writer, sheet_name="README", index=False)
    data_check.to_excel(writer, sheet_name="Data_Check", index=False)
    missingness.to_excel(writer, sheet_name="Feature_Missingness", index=False)

    model_df.to_excel(writer, sheet_name="Model_Dataset", index=False)

    label_distribution.to_excel(writer, sheet_name="Label_3Class_Distribution", index=False)
    binary_label_distribution.to_excel(writer, sheet_name="Label_Binary_Distribution", index=False)

    overall_perf.to_excel(writer, sheet_name="Overall_Performance", index=False)
    perf_by_split.to_excel(writer, sheet_name="Performance_By_Split", index=False)
    perf_by_quality.to_excel(writer, sheet_name="Performance_By_Label", index=False)
    quality_by_session.to_excel(writer, sheet_name="Label_By_Session", index=False)

    pd.DataFrame({"feature": numeric_features, "type": "numeric"}).to_excel(
        writer, sheet_name="Numeric_Features", index=False
    )
    pd.DataFrame({"feature": categorical_features, "type": "categorical"}).to_excel(
        writer, sheet_name="Categorical_Features", index=False
    )

    metrics_df.to_excel(writer, sheet_name="Model_Metrics", index=False)

    for name, fi in feature_importance_tables.items():
        sheet = ("FI_" + name)[:31]
        fi.to_excel(writer, sheet_name=sheet, index=False)

    for name, perm in permutation_tables.items():
        sheet = ("Permutation_" + name)[:31]
        perm.to_excel(writer, sheet_name=sheet, index=False)

    shap_summary.to_excel(writer, sheet_name="SHAP_Optional", index=False)

    for name, bucket_perf in score_tables.items():
        sheet = ("Score_Buckets_" + name)[:31]
        bucket_perf.to_excel(writer, sheet_name=sheet, index=False)

    if not scored_all.empty:
        scored_all.to_excel(writer, sheet_name="Scored_All_Models", index=False)

    conclusion_df.to_excel(writer, sheet_name="AI_Validation_Conclusion", index=False)

print("\nDone. Exported:", OUTPUT_FILE)
files.download(OUTPUT_FILE)


Please upload the event dataset.
Recommended: 0050_v1_1_ai_event_dataset_full_features.csv
Acceptable: xlsx with Combined_Data sheet.


Saving 0050_v1_1_ai_event_dataset_full_features.csv to 0050_v1_1_ai_event_dataset_full_features.csv
Loaded: 0050_v1_1_ai_event_dataset_full_features.csv
Raw shape: (361, 51)
Columns:
['trade_id', 'entry_time', 'exit_time', 'entry_date', 'entry_year', 'entry_month', 'entry_weekday', 'entry_time_hhmm', 'data_split', 'session_id', 'session_block', 'is_2025_high_prob_session', 'minutes_since_open', 'minutes_to_close', 'signal_count_today', 'entry_price', 'exit_price', 'qty', 'position_value_twd', 'pnl_twd', 'pnl_pct', 'mfe_twd', 'mfe_pct', 'mae_twd', 'mae_pct', 'cum_pnl_twd', 'cum_pnl_pct', 'bars_held', 'holding_minutes', 'exit_reason', 'exit_relative_deviation', 'entry_relative_deviation', 'volume_ratio_at_entry', 'deviation_slope_1bar', 'deviation_slope_3bar', 'etf_return_5m', 'etf_return_15m', 'tw50_return_5m', 'tw50_return_15m', 'realized_vol_10bar', 'rd_bucket', 'volume_bucket', 'slope_1bar_bucket', 'tw50_5m_direction', 'profit_flag', 'label_prelim', 'label_id_prelim', 'quality_label'

,trade_id,entry_time,exit_time,entry_date,entry_year,entry_month,entry_weekday,entry_time_hhmm,data_split,session_id,...,volume_bucket,slope_1bar_bucket,tw50_5m_direction,profit_flag,label_prelim,label_id_prelim,quality_label,quality_label_id,entry_feature_string,exit_feature_string
0,1,2022-06-01 09:40,2022-06-01 09:50,2022-06-01,2022,6,Wed,09:40,Train_探索_2022_2024,2,...,2 ~ 3,惡化中_負斜率,上行,0,Weak_Flat_暫定,2,Weak_Flat_or_NoMove,1,ENTRY|RD=-0.2445|VR=2.1142|S1=-0.0724|S3=-0.13...,EXIT_TIME|XRD=-0.1218|BH=2
1,2,2022-06-01 10:25,2022-06-01 10:35,2022-06-01,2022,6,Wed,10:25,Train_探索_2022_2024,4,...,2 ~ 3,惡化中_負斜率,上行,0,Weak_Flat_暫定,2,Weak_Flat_or_NoMove,1,ENTRY|RD=-0.2473|VR=2.2161|S1=-0.0665|S3=-0.27...,EXIT_TIME|XRD=-0.1656|BH=2
2,3,2022-06-14 12:05,2022-06-14 12:15,2022-06-14,2022,6,Tue,12:05,Train_探索_2022_2024,4,...,2 ~ 3,近乎持平,上行,1,Strong_Profit_暫定,1,Strong_Profit_NonReversion,2,ENTRY|RD=-0.2884|VR=2.0584|S1=-0.001|S3=-0.137...,EXIT_TIME|XRD=-0.1405|BH=2


Normalized data_split counts:
data_split
Train_2022_2024      174
Validation_2025      117
Recent_Check_2026     70
Name: count, dtype: int64
Numeric features: ['entry_relative_deviation', 'deviation_slope_1bar', 'deviation_slope_3bar', 'tw50_return_5m', 'tw50_return_15m', 'etf_return_5m', 'etf_return_15m', 'volume_ratio_at_entry', 'realized_vol_10bar', 'signal_count_today', 'entry_minutes', 'minutes_since_open', 'minutes_to_close']
Categorical features: ['session_block', 'entry_weekday']
After split filter: (361, 57)
data_split
Train_2022_2024      174
Validation_2025      117
Recent_Check_2026     70
Name: count, dtype: int64
Split sizes: {'train': 174, 'validation': 117, 'recent': 70}

Data Check


,item,value
0,input_file,0050_v1_1_ai_event_dataset_full_features.csv
1,raw_rows,361
2,rows_after_filter,361
3,train_rows,174
4,validation_rows,117
5,recent_rows,70
6,pnl_col,pnl_twd
7,round_trip_cost_for_label,0.02
8,positive_label_definition,good_signal_cost002 = 1 if pnl_twd - 0.02 > 0
9,numeric_features,"entry_relative_deviation, deviation_slope_1bar..."


...


Score Bucket Performance - XGBoost_Optional


,model,dataset,score_bucket,trades,wins,losses,flats,win_rate,net_pnl,avg_pnl,gross_profit,gross_loss,profit_factor
0,XGBoost_Optional,Recent_Check_2026,"(0.065, 0.183]",14,10,4,0,0.714286,1.02,0.072857,1.45,-0.43,3.372093
1,XGBoost_Optional,Recent_Check_2026,"(0.183, 0.294]",14,6,8,0,0.428571,-0.58,-0.041429,0.53,-1.11,0.477477
2,XGBoost_Optional,Recent_Check_2026,"(0.294, 0.434]",14,5,9,0,0.357143,0.27,0.019286,0.80,-0.53,1.509434
3,XGBoost_Optional,Recent_Check_2026,"(0.434, 0.594]",14,10,4,0,0.714286,1.77,0.126429,2.00,-0.23,8.695652
4,XGBoost_Optional,Recent_Check_2026,"(0.594, 0.919]",14,9,5,0,0.642857,0.12,0.008571,0.92,-0.80,1.150000
5,XGBoost_Optional,Train_2022_2024,"(0.015299999999999998, 0.0616]",35,0,35,0,0.000000,-1.25,-0.035714,0.00,-1.25,0.000000
6,XGBoost_Optional,Train_2022_2024,"(0.0616, 0.118]",35,0,35,0,0.000000,-1.35,-0.038571,0.00,-1.35,0.000000
7,XGBoost_Optional,Train_2022_2024,"(0.118, 0.184]",34,0,34,0,0.000000,-1.48,-0.043529,0.00,-1.48,0.000000
8,XGBoost_Optional,Train_2022_2024,"(0.184, 0.704]",35,18,17,0,0.514286,0.10,0.002857,0.74,-0.64,1.156250
9,XGBoost_Optional,Train_2022_2024,"(0.704, 0.902]",35,35,0,0,1.000000,2.05,0.058571,2.05,0.00,inf



Done. Exported: 0050_ai_signal_quality_validation_outputs_FIXED.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>